# 08 — Recruitment Case Study & Executive Summary

This notebook converts the analytical pipeline into a **club-style recruitment case study**.

It combines:

- **player value outputs** from xT and VAEP
- **team tactical profiles**
- **player–team fit scores**

to generate a practical shortlist and an executive-style summary for decision makers.

## Objectives

1. Load the outputs from Notebooks 05, 06, and 07.
2. Select a target club/team for the case study.
3. Define recruitment filters and shortlist rules.
4. Rank candidate players by:
   - tactical fit
   - player value
   - progression contribution
   - efficiency
5. Build an executive summary table and visual outputs.
6. Persist the final shortlist for downstream reporting.

## Outputs

- `data/processed/recruitment_case_shortlist.parquet`
- `data/processed/recruitment_case_summary.parquet`
- `db/football_value.duckdb`
  - `recruitment_case_shortlist`
  - `recruitment_case_summary`


## 0. Setup

In [ ]:
from __future__ import annotations

import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
warnings.filterwarnings("ignore")


In [ ]:
try:
    import duckdb
except Exception as e:
    raise ImportError("duckdb is required. Install it with: pip install duckdb") from e

try:
    from sklearn.preprocessing import MinMaxScaler
    SKLEARN_AVAILABLE = True
except Exception:
    SKLEARN_AVAILABLE = False


## 1. Project paths

In [ ]:
def find_repo_root(start: Optional[Path] = None) -> Path:
    start = start or Path.cwd()
    for p in [start, *start.parents]:
        if (p / "README.md").exists():
            return p
    return start

REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
DB_DIR = REPO_ROOT / "db"
REPORTS_DIR = REPO_ROOT / "reports"

for d in [PROCESSED_DIR, DB_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DB_PATH = DB_DIR / "football_value.duckdb"

PLAYER_VALUE_PATH = PROCESSED_DIR / "player_value_table.parquet"
TEAM_TACTICAL_PATH = PROCESSED_DIR / "team_tactical_table.parquet"
PLAYER_TEAM_FIT_PATH = PROCESSED_DIR / "player_team_fit.parquet"

SHORTLIST_PATH = PROCESSED_DIR / "recruitment_case_shortlist.parquet"
SUMMARY_PATH = PROCESSED_DIR / "recruitment_case_summary.parquet"

REPO_ROOT, DB_PATH


## 2. Configuration

In [ ]:
@dataclass(frozen=True)
class Config:
    target_team: str = "Barcelona"
    top_n_shortlist: int = 15
    min_player_actions: int = 150
    exclude_same_team: bool = True
    save_outputs: bool = True

CFG = Config()
CFG


## 3. Load required datasets

Expected inputs:
- `player_value_table.parquet`
- `team_tactical_table.parquet`
- `player_team_fit.parquet`


In [ ]:
required_paths = [
    PLAYER_VALUE_PATH,
    TEAM_TACTICAL_PATH,
    PLAYER_TEAM_FIT_PATH,
]

missing_paths = [str(p) for p in required_paths if not p.exists()]
if missing_paths:
    raise FileNotFoundError(
        "Missing required inputs. Run Notebooks 05, 06, and 07 first:\n" + "\n".join(missing_paths)
    )

player_value = pd.read_parquet(PLAYER_VALUE_PATH)
team_tactical = pd.read_parquet(TEAM_TACTICAL_PATH)
player_team_fit = pd.read_parquet(PLAYER_TEAM_FIT_PATH)

print("player_value:", player_value.shape)
print("team_tactical:", team_tactical.shape)
print("player_team_fit:", player_team_fit.shape)


## 4. Select the target team

In [ ]:
available_teams = sorted(team_tactical["team"].dropna().unique().tolist())
print("Available teams:", available_teams[:20], "..." if len(available_teams) > 20 else "")

if CFG.target_team not in available_teams:
    raise ValueError(
        f"Target team '{CFG.target_team}' not found. Choose one of: {available_teams}"
    )

team_profile = team_tactical[team_tactical["team"] == CFG.target_team].copy()
team_profile.T


## 5. Build recruitment candidate pool

In [ ]:
candidate_pool = (
    player_team_fit[player_team_fit["team"] == CFG.target_team]
    .merge(
        player_value[[
            "player",
            "primary_team",
            "n_actions",
            "n_matches",
            "xt_per_90",
            "vaep_per_90",
            "xt_per_100_actions",
            "vaep_per_100_actions",
            "progressive_actions_per_90",
            "progression_score",
            "efficiency_score",
            "overall_value_score",
            "scouting_score",
            "rule_based_segment",
        ]],
        on="player",
        how="left"
    )
    .copy()
)

candidate_pool = candidate_pool[candidate_pool["n_actions"] >= CFG.min_player_actions].copy()

if CFG.exclude_same_team:
    candidate_pool = candidate_pool[candidate_pool["primary_team"] != CFG.target_team].copy()

candidate_pool.head()


## 6. Build recruitment ranking score

In [ ]:
ranking_cols = [
    "fit_score_scaled",
    "scouting_score",
    "vaep_per_90",
    "xt_per_90",
    "progressive_actions_per_90",
]

rank_df = candidate_pool[ranking_cols].copy()

if SKLEARN_AVAILABLE:
    scaler = MinMaxScaler()
    scaled = scaler.fit_transform(rank_df.fillna(0))
    scaled_df = pd.DataFrame(scaled, columns=[f"{c}_scaled" for c in ranking_cols], index=candidate_pool.index)
else:
    scaled_df = rank_df.rank(pct=True).rename(columns={c: f"{c}_scaled" for c in rank_df.columns})

candidate_pool = pd.concat([candidate_pool, scaled_df], axis=1)

candidate_pool["recruitment_score"] = (
    0.35 * candidate_pool["fit_score_scaled"]
    + 0.25 * candidate_pool["scouting_score_scaled"]
    + 0.20 * candidate_pool["vaep_per_90_scaled"]
    + 0.10 * candidate_pool["xt_per_90_scaled"]
    + 0.10 * candidate_pool["progressive_actions_per_90_scaled"]
)

candidate_pool = candidate_pool.sort_values(
    ["recruitment_score", "fit_score_scaled", "scouting_score"],
    ascending=[False, False, False]
).reset_index(drop=True)

candidate_pool.head(20)


## 7. Final shortlist

In [ ]:
shortlist = candidate_pool.head(CFG.top_n_shortlist).copy()

shortlist = shortlist[[
    "player",
    "primary_team",
    "team",
    "team_style",
    "player_segment",
    "fit_score_scaled",
    "scouting_score",
    "xt_per_90",
    "vaep_per_90",
    "progressive_actions_per_90",
    "efficiency_score",
    "overall_value_score",
    "recruitment_score",
]].copy()

shortlist


## 8. Executive summary table

In [ ]:
case_summary = pd.DataFrame({
    "metric": [
        "target_team",
        "target_team_style",
        "candidate_pool_size",
        "shortlist_size",
        "avg_fit_score_shortlist",
        "avg_recruitment_score_shortlist",
        "avg_vaep_per_90_shortlist",
        "avg_xt_per_90_shortlist",
        "avg_progressive_actions_per_90_shortlist",
    ],
    "value": [
        str(CFG.target_team),
        str(team_profile["rule_based_style"].iloc[0]),
        int(len(candidate_pool)),
        int(len(shortlist)),
        round(float(shortlist["fit_score_scaled"].mean()), 4),
        round(float(shortlist["recruitment_score"].mean()), 4),
        round(float(shortlist["vaep_per_90"].mean()), 4),
        round(float(shortlist["xt_per_90"].mean()), 4),
        round(float(shortlist["progressive_actions_per_90"].mean()), 4),
    ]
})

# Make parquet export robust: mixed values -> string
case_summary["value"] = case_summary["value"].astype(str)

case_summary

## 9. Visualise shortlist value vs fit

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(
    shortlist["fit_score_scaled"],
    shortlist["scouting_score"],
    s=70
)

for _, r in shortlist.iterrows():
    ax.annotate(r["player"], (r["fit_score_scaled"], r["scouting_score"]), fontsize=8)

ax.set_title(f"Recruitment Case Study — {CFG.target_team}: Fit vs Scouting Score")
ax.set_xlabel("Fit score (scaled)")
ax.set_ylabel("Scouting score")
plt.show()


## 10. Visualise shortlist xT vs VAEP

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(
    shortlist["xt_per_90"],
    shortlist["vaep_per_90"],
    s=70
)

for _, r in shortlist.iterrows():
    ax.annotate(r["player"], (r["xt_per_90"], r["vaep_per_90"]), fontsize=8)

ax.set_title(f"Recruitment Case Study — {CFG.target_team}: xT vs VAEP")
ax.set_xlabel("xT per 90")
ax.set_ylabel("VAEP per 90")
plt.show()


## 11. Recruitment narrative view

In [ ]:
recruitment_narrative = shortlist.copy()

recruitment_narrative["recommendation_label"] = np.select(
    [
        (recruitment_narrative["fit_score_scaled"] >= 0.80) & (recruitment_narrative["scouting_score"] >= 0.80),
        (recruitment_narrative["fit_score_scaled"] >= 0.70),
        (recruitment_narrative["scouting_score"] >= 0.75),
    ],
    [
        "High-priority target",
        "Strong tactical fit",
        "High-value alternative",
    ],
    default="Context-dependent option"
)

recruitment_narrative[[
    "player",
    "primary_team",
    "player_segment",
    "fit_score_scaled",
    "scouting_score",
    "recruitment_score",
    "recommendation_label",
]].head(20)


## 12. Save outputs

In [ ]:
if CFG.save_outputs:
    shortlist.to_parquet(SHORTLIST_PATH, index=False)
    case_summary.to_parquet(SUMMARY_PATH, index=False)

    try:
        con.close()
    except:
        pass

    con = duckdb.connect(str(DB_PATH))
    con.execute("CREATE OR REPLACE TABLE recruitment_case_shortlist AS SELECT * FROM read_parquet(?)", [str(SHORTLIST_PATH)])
    con.execute("CREATE OR REPLACE TABLE recruitment_case_summary AS SELECT * FROM read_parquet(?)", [str(SUMMARY_PATH)])
    con.close()

    print(f"Saved shortlist to: {SHORTLIST_PATH}")
    print(f"Saved summary to:   {SUMMARY_PATH}")
    print(f"Updated DuckDB at:  {DB_PATH}")


## 13. Preview from DuckDB

In [ ]:
con = duckdb.connect(str(DB_PATH))

summary_sql = '''
SELECT
    COUNT(*) AS shortlist_size,
    AVG(fit_score_scaled) AS avg_fit_score,
    AVG(recruitment_score) AS avg_recruitment_score,
    AVG(vaep_per_90) AS avg_vaep_per_90,
    AVG(xt_per_90) AS avg_xt_per_90
FROM recruitment_case_shortlist
'''
preview = con.execute(summary_sql).df()
con.close()

preview


## Recruitment Case Study — Barcelona

### Tactical profile

The tactical profiling model identifies **Barcelona** as a **Control-Possession side**.

Key characteristics:

* High **control style score** (0.87)
* Strong **final third value creation**
* Low directness profile
* High positional play and structured buildup

This profile prioritizes players capable of:

* Maintaining possession
* Progressing the ball through structured phases
* Creating value in the final third through passing and movement

---

### Candidate pool

The analysis considered:

* **334 players**
* **51 team tactical profiles**
* **16,700 player–team compatibility pairs**

From this space, a **shortlist of 15 candidates** was generated.

Average shortlist characteristics:

| Metric                    | Value     |
| ------------------------- | --------- |
| Average fit score         | **0.746** |
| Average recruitment score | **0.731** |
| Average VAEP / 90         | **19.72** |
| Average xT / 90           | **0.164** |

These values indicate a candidate pool with both **high tactical compatibility** and **strong value creation metrics**.

---

### Top recruitment targets

The highest ranked players combine **tactical fit** and **individual value creation**.

Key candidates include:

* Mesut Özil
* Neymar Jr
* Toni Kroos
* Luis Suárez
* Jérôme Boateng

These players show:

* High **scouting score**
* Strong **fit with possession-oriented structures**
* High **expected possession value generation**

---

### Performance profile analysis

The xT vs VAEP plot highlights two different player archetypes:

**High VAEP creators**

* Mesut Özil
* Neymar Jr
* Toni Kroos

These players contribute strongly to:

* chance creation
* attacking sequence value

**High xT progressors**

* Luis Suárez
* Jérôme Boateng
* Neymar Jr

These profiles combine:

* ball progression
* threat creation

---

### Recruitment strategy interpretation

The model suggests Barcelona should prioritize:

1️⃣ **Elite creative controllers**

Players capable of maintaining possession dominance while creating attacking value.

Examples:

* Özil
* Kroos

2️⃣ **High-impact attacking creators**

Players with elite VAEP contribution.

Example:

* Neymar

3️⃣ **Balanced possession facilitators**

Players capable of contributing both progression and structure.

Example:

* Kimmich

---

### Key insight

The shortlist is strongly biased toward **elite high-volume creators**, which aligns with Barcelona's tactical identity as a **possession-dominant team**.

This validates that the **team-style compatibility model successfully captures tactical requirements**.
